# P03 — Differential Privacy con PyTorch y Opacus

Velora Finance quiere entrenar un modelo de riesgo reduciendo la influencia observable de cada registro individual en el modelo publicado.

La práctica compara:

1. un modelo PyTorch convencional;
2. el mismo tipo de modelo entrenado con **DP-SGD mediante Opacus**.

El objetivo es observar el trade-off entre **privacidad y utilidad**, no aprender PyTorch. El código está preparado y puede ejecutarse en Google Colab con GPU T4.


## 1. Activar GPU T4

En Colab: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**.

El notebook funcionará también en CPU, pero la GPU permite ejecutar las comparaciones con más margen.


In [ ]:
!pip -q install opacus==1.6.0


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import urllib.request
import copy
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score
from opacus import PrivacyEngine

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Dataset

Para DP-SGD utilizamos una cohorte sintética mayor generada con el mismo proceso que Velora. El generador ya admite cambiar `n`, de modo que mantenemos el caso y las variables pero contamos con suficientes ejemplos para que el entrenamiento privado sea más representativo.


In [ ]:
GENERATOR_URL = "https://raw.githubusercontent.com/pacosepulveda/ia-confiable-explicable-labs/main/resources/generate_velora_credit.py"
namespace = {"__name__": "velora_generator"}
with urllib.request.urlopen(GENERATOR_URL) as response:
    exec(response.read().decode("utf-8"), namespace)

df = namespace["generate_dataset"](n=30000, seed=42)
print(df.shape)
print(f"Tasa de impago: {df['default'].mean():.2%}")


## 3. Preparar train/test

Usamos las mismas variables de P01 y P02. El escalado se ajusta únicamente con train.


In [ ]:
FEATURES = [
    "age", "annual_income", "debt_ratio", "employment_months",
    "late_payments_12m", "credit_history_months", "existing_debt",
    "requested_amount", "savings_balance",
]
X_train, X_test, y_train, y_test = train_test_split(
    df[FEATURES], df["default"], test_size=0.20,
    random_state=42, stratify=df["default"]
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype("float32")
X_test_scaled = scaler.transform(X_test).astype("float32")
y_train_np = y_train.to_numpy(dtype="int64")
y_test_np = y_test.to_numpy(dtype="int64")

train_ds = TensorDataset(
    torch.tensor(X_train_scaled), torch.tensor(y_train_np)
)
test_x = torch.tensor(X_test_scaled, device=device)
print("Train:", len(train_ds), "Test:", len(y_test_np))


## 4. Modelo

Usamos una red muy pequeña. No buscamos maximizar rendimiento; necesitamos un modelo suficientemente real para comparar entrenamiento convencional y DP-SGD.


In [ ]:
def make_model():
    return nn.Sequential(
        nn.Linear(len(FEATURES), 32),
        nn.ReLU(),
        nn.Linear(32, 16),
        nn.ReLU(),
        nn.Linear(16, 2),
    )

def evaluate(model):
    model.eval()
    with torch.no_grad():
        logits = model(test_x)
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        pred = (probs >= 0.5).astype(int)
    return {
        "roc_auc": roc_auc_score(y_test_np, probs),
        "accuracy": accuracy_score(y_test_np, pred),
        "recall": recall_score(y_test_np, pred, zero_division=0),
    }

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(yb)
    return total_loss / len(loader.dataset)


## 5. Entrenamiento convencional

Primero obtenemos una referencia sin Differential Privacy.


In [ ]:
BATCH_SIZE = 512
EPOCHS = 8

torch.manual_seed(42)
baseline_model = make_model().to(device)
baseline_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
baseline_optimizer = torch.optim.Adam(baseline_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    loss = train_epoch(
        baseline_model, baseline_loader,
        baseline_optimizer, criterion
    )

baseline_metrics = evaluate(baseline_model)
print("Modelo convencional")
for k, v in baseline_metrics.items():
    print(f"{k:10}: {v:.3f}")


## 6. Entrenamiento privado con Opacus

Opacus aplica DP-SGD: clipping por ejemplo, ruido calibrado y contabilidad del presupuesto.

Selecciona un `TARGET_EPSILON`. Un valor menor representa una garantía de privacidad más fuerte, normalmente a costa de más ruido o menor utilidad.


In [ ]:
TARGET_EPSILON = 3.0 #@param {type:"slider", min:0.5, max:10.0, step:0.5}
TARGET_DELTA = 1e-5
MAX_GRAD_NORM = 1.0

torch.manual_seed(42)
private_model = make_model().to(device)
private_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
private_optimizer = torch.optim.Adam(private_model.parameters(), lr=1e-3)

privacy_engine = PrivacyEngine(accountant="prv")
private_model, private_optimizer, private_loader = privacy_engine.make_private_with_epsilon(
    module=private_model,
    optimizer=private_optimizer,
    data_loader=private_loader,
    target_epsilon=TARGET_EPSILON,
    target_delta=TARGET_DELTA,
    epochs=EPOCHS,
    max_grad_norm=MAX_GRAD_NORM,
)

for epoch in range(EPOCHS):
    loss = train_epoch(
        private_model, private_loader,
        private_optimizer, criterion
    )

private_metrics = evaluate(private_model)
spent_epsilon = privacy_engine.get_epsilon(TARGET_DELTA)

print("Modelo con DP-SGD")
print(f"epsilon gastado: {spent_epsilon:.3f}")
print(f"delta          : {TARGET_DELTA}")
for k, v in private_metrics.items():
    print(f"{k:10}: {v:.3f}")


### Qué significa `(ε, δ)`

La garantía no dice “nadie puede reconstruir ningún dato”. Formaliza que la distribución de resultados del mecanismo cambia de forma limitada cuando un único ejemplo entra o sale del conjunto de entrenamiento.

`epsilon` y `delta` deben interpretarse junto con el mecanismo, el tamaño del dataset, el número de pasos y el modelo de amenazas. No existe un valor universalmente correcto.


## 7. Comparar utilidad


In [ ]:
comparison = pd.DataFrame([
    {"modelo": "Sin DP", **baseline_metrics},
    {"modelo": f"DP-SGD ε≈{spent_epsilon:.2f}", **private_metrics},
]).set_index("modelo")
display(comparison.style.format("{:.3f}"))


### Interpretación

La comparación hace visible el coste de la garantía de privacidad. Dependiendo de `epsilon`, número de epochs y clipping, la pérdida de utilidad puede ser pequeña o considerable.

No buscamos “el epsilon ganador”. Buscamos una configuración que pueda justificarse respecto al riesgo de privacidad y a la utilidad mínima necesaria.


## 8. Repetir con distintos presupuestos

Ejecuta de nuevo la sección privada con, por ejemplo:

- `ε = 1`
- `ε = 3`
- `ε = 8`

Registra qué ocurre con ROC-AUC y Recall. El coste no tiene por qué ser lineal.


## 9. Conexión con fairness

Una técnica de privacidad puede cambiar el rendimiento de forma diferente entre subgrupos. Como profundización, comparamos Recall por `customer_group` para el modelo convencional y el privado.


In [ ]:
test_groups = df.loc[X_test.index, "customer_group"].to_numpy()

def probabilities(model):
    model.eval()
    with torch.no_grad():
        return torch.softmax(model(test_x), dim=1)[:, 1].cpu().numpy()

base_probs = probabilities(baseline_model)
priv_probs = probabilities(private_model)

rows = []
for g in sorted(np.unique(test_groups)):
    mask = test_groups == g
    for name, probs in [("Sin DP", base_probs), ("DP-SGD", priv_probs)]:
        pred = (probs[mask] >= 0.5).astype(int)
        rows.append({
            "grupo": g,
            "modelo": name,
            "recall_impago": recall_score(
                y_test_np[mask], pred, zero_division=0
            ),
            "roc_auc": roc_auc_score(y_test_np[mask], probs[mask]),
        })

group_utility = pd.DataFrame(rows)
display(group_utility.pivot(index="grupo", columns="modelo", values=["recall_impago", "roc_auc"]).style.format("{:.3f}"))


### Idea clave

Privacidad y fairness pueden interactuar. Añadir una garantía de privacidad no garantiza que la pérdida de utilidad se distribuya de manera uniforme entre grupos. Por eso las dimensiones de confiabilidad deben evaluarse conjuntamente.


## 10. Federated Learning: qué aporta y qué no

Federated Learning reduce la necesidad de centralizar datos brutos: el entrenamiento se ejecuta en clientes o silos y se intercambian actualizaciones.

Pero FL **no proporciona por sí solo Differential Privacy**. Las actualizaciones pueden filtrar información y el sistema sigue necesitando controles frente a inferencia, poisoning, clientes maliciosos, metadatos y compromiso del agregador.

Flower es una herramienta real para implementar y simular federaciones, incluida aceleración GPU. No la ejecutamos aquí porque el objetivo de esta práctica es medir una garantía de privacidad formal; introducir la arquitectura Flower desviaría el foco hacia infraestructura de FL.


# Conclusiones de P03

- Differential Privacy puede aplicarse al entrenamiento real, no solo a estadísticas agregadas.
- Opacus implementa DP-SGD y contabiliza el presupuesto de privacidad durante entrenamiento.
- La T4 acelera un experimento real de PyTorch, pero no es el objetivo pedagógico.
- Privacidad implica un trade-off medible con utilidad.
- `epsilon` no debe interpretarse como una nota de 0 a 10 ni seleccionarse por una regla universal.
- La pérdida de utilidad puede ser diferente entre grupos, conectando privacidad con fairness.
- Federated Learning y Differential Privacy son complementarios, pero resuelven problemas distintos.
